In [ ]:
# Install compatible PyTorch
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121

# Install xformers compatible with torch 2.3
!pip install xformers==0.0.27

# Install unsloth
!pip install unsloth

40

In [3]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen3.5-0.8B",
    load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [4]:
from datasets import load_dataset  ,concatenate_datasets 
import random  

SEED = 42 
TARGET_DATASET = 100000
def get_load_dataset(): 
    ds1 = load_dataset("Rohit-Katkar2003/Instruction-dataset",   split="train")
    ds2 = load_dataset("Rohit-Katkar2003/summarization-dataset", split="train")
    ds3 = load_dataset("Rohit-Katkar2003/Extractional-dataset",  split="train")
    ds4 = load_dataset("Rohit-Katkar2003/decomposition-dataset", split="train") 
    return ds1 , ds2 , ds3 , ds4 


## applying the weights of dataset 
WEIGHTS = {
    "instruction":   0.35,
    "summarization": 0.30,
    "extraction":    0.20,
    "decomposition": 0.15,
} 

def weights_of_dataset(ds1,ds2,ds3,ds4): 
    random.seed(SEED)

    source_map = {
        "instruction":  ds1,
        "summarization": ds2,
        "extraction":   ds3,
        "decomposition": ds4,
    } 
    selected_split = [] 
    for name ,ds in source_map.items(): 
        target_n = int(TARGET_DATASET * WEIGHTS[name]) 
        
        if len(ds) >= target_n:
            idx =  random.sample(range(len(ds)) , target_n) 
            split = ds.select(idx) 
            
        else: 
            repeats = (target_n // len(ds)) + 1
            idx = (list(range(len(ds))) * repeats)[:target_n]
            random.shuffle(idx)
            split = ds.select(idx) 
            
        selected_split.append(split) 
        
    combined_dataset = concatenate_datasets(selected_split) 
    combined = combined_dataset.shuffle(seed=SEED) 
    return combined 



In [5]:
import pandas as pd

DS1 , DS2 , DS3 , DS4 = get_load_dataset() 

DATASET = weights_of_dataset(DS1 , DS2 , DS3 , DS4) 

ds = pd.DataFrame(DATASET) 

In [6]:
ds['task'].value_counts()

task
instruction_tune          35000
summarization             30000
information_extraction    20000
query_decomposition       15000
Name: count, dtype: int64

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

In [7]:
model = FastLanguageModel.get_peft_model(
    model , 
    finetune_vision_layers = False , 
    finetune_language_layers = True , # as you know what we focus on **only text**
    finetune_attention_modules=True , 
    finetune_mlp_modules = True , 

    r = 16   ,  ## if it is larger high accuracy but might overfit 
    lora_alpha = 16  , # r=lora-alpha 
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [8]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3_5ForConditionalGeneration(
      (model): Qwen3_5Model(
        (visual): Qwen3_5VisionModel(
          (patch_embed): Qwen3_5VisionPatchEmbed(
            (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 768)
          (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-11): 12 x Qwen3_5VisionBlock(
              (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
              (attn): Qwen3_5VisionAttention(
                (qkv): Linear(in_features=768, out_features=2304, bias=True)
                (proj): Linear(in_features=768, out_features=768, bias=True)
              )
              (mlp): Qwen3_5VisionMLP(
                (linear_fc1): Linear(in_features=768, out_features=3072, bias=True)
                (linear

In [9]:
ds.head()

,dataset,task,system,messages
0,cnn_dailymail,summarization,You are a research summarization assistant. Yo...,"[{'role': 'user', 'content': 'Provide a dense,..."
1,cnn_dailymail,summarization,You are a research summarization assistant. Yo...,"[{'role': 'user', 'content': 'Write a brief, a..."
2,alpaca,instruction_tune,You are Helpful assistant.,"[{'role': 'user', 'content': 'Name some instit..."
3,synthetic_decomposition,query_decomposition,You are a research planning agent. Break compl...,"[{'role': 'user', 'content': 'Decompose this r..."
4,synthetic_decomposition,query_decomposition,You are a research planning agent. Break compl...,"[{'role': 'user', 'content': 'Decompose this r..."


In [10]:
def format_to_chatml(sample):
    system   = (sample.get("system") or "You are a helpful AI assistant.").strip()
    messages = sample.get("messages") or []

    parts = [f"<|im_start|>system\n{system}<|im_end|>"]
    for msg in messages:
        role    = (msg.get("role")    or "").strip()
        content = (msg.get("content") or "").strip()
        if role in ("user", "assistant") and content:
            parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")

    return {"text": "\n".join(parts) + "\n"}  
